In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
location = 'http://localhost:6333'

In [3]:
from qdrant_client import QdrantClient

client = QdrantClient(location=location)

/Users/andreicristea/personal/qdrant-interview/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from qdrant_client.models import Distance, VectorParams

client.create_collection(
    collection_name='test_collection',
    vectors_config=VectorParams(size=4, distance=Distance.DOT)
)

UnexpectedResponse: Unexpected Response: 409 (Conflict)
Raw response content:
b'{"status":{"error":"Wrong input: Collection `test_collection` already exists!"},"time":0.003158672}'

In [6]:
from qdrant_client.models import PointStruct

operation_info = client.upsert(
    collection_name="test_collection",
    wait=True,
    points=[
        PointStruct(id=1, vector=[0.05, 0.61, 0.76, 0.74], payload={"city": "Berlin"}),
        PointStruct(id=2, vector=[0.19, 0.81, 0.75, 0.11], payload={"city": "London"}),
        PointStruct(id=3, vector=[0.36, 0.55, 0.47, 0.94], payload={"city": "Moscow"}),
        PointStruct(id=4, vector=[0.18, 0.01, 0.85, 0.80], payload={"city": "New York"}),
        PointStruct(id=5, vector=[0.24, 0.18, 0.22, 0.44], payload={"city": "Beijing"}),
        PointStruct(id=6, vector=[0.35, 0.08, 0.11, 0.44], payload={"city": "Mumbai"}),
    ],
)


In [5]:
print(operation_info)


NameError: name 'operation_info' is not defined

In [95]:
search_result = client.query_points(
    collection_name="test_collection",
    query=[0.2, 0.1, 0.9, 0.7],
    with_payload=False,
    limit=3
).points

print(search_result)

[ScoredPoint(id=4, version=4, score=1.362, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=1, version=4, score=1.273, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=3, version=4, score=1.208, payload=None, vector=None, shard_key=None, order_value=None)]


In [6]:
from qdrant_client.models import FieldCondition, Filter, MatchValue


search_result = client.query_points(
    collection_name='test_collection',
    query=[0.2, 0.1, 0.9, 0.7],
    query_filter=Filter(
        must=[FieldCondition(key='city', match=MatchValue(value="London"))]
    )
)

In [7]:
search_result

QueryResponse(points=[ScoredPoint(id=2, version=4, score=0.871, payload={'city': 'London'}, vector=None, shard_key=None, order_value=None)])

# Load the dataset and load embeddings

In [8]:
from enum import StrEnum

class DatasetSrc(StrEnum):
    WIKI_VOYAGE_EU = "ashmib/wikivoyage-eu-city-embeddings"
    LOCAL_VOYAGE_LISTINGS = '../datasets/wikivoyage-listings-en.csv'

In [9]:
from datasets import load_dataset

wiki_voyage_eu  = load_dataset(DatasetSrc.WIKI_VOYAGE_EU)

In [10]:
wiki_voyage_eu

DatasetDict({
    train: Dataset({
        features: ['city', 'country', 'lat', 'lng', 'population', 'abstract', 'combined', 'embedding'],
        num_rows: 160
    })
})

In [11]:
wiki_voyage_eu.keys()

dict_keys(['train'])

In [12]:
wiki_voyage_eu = wiki_voyage_eu['train']

In [13]:
wiki_voyage_eu[0]

{'city': 'Aalborg',
 'country': 'Denmark',
 'lat': 57.05,
 'lng': 9.9167,
 'population': 143598.0,
 'abstract': 'Aalborg is the largest city in North Jutland, Denmark. Its population, as of 2016, is 134,672, making it the fourth largest city in Denmark.',
 'combined': 'city: Aalborg, country: Denmark, population: 143598.0, abstract: Aalborg is the largest city in North Jutland, Denmark. Its population, as of 2016, is 134,672, making it the fourth largest city in Denmark.',
 'embedding': '[-0.0032697843853384256, 0.007246419321745634, -0.009926767088472843, 0.028170665726065636, -0.013943376019597054, -0.009734532795846462, 0.009892424568533897, -0.00396350584924221, 0.011268466711044312, 0.029671311378479004, 0.02826191671192646, -0.020326530560851097, -0.0018195175798609853, -0.018696848303079605, -0.008758136071264744, -0.03397727385163307, -0.04039132222533226, -0.03460145369172096, -0.019473925232887268, -0.017622580751776695, -0.03790139779448509, -0.008759676478803158, -0.0715630

### Create collection with embedding configuration for the cities infodump

In [17]:
class EmbeddingNames(StrEnum):
    DENSE_BAAI_384 = 'BAAI/bge-small-en'
    DENSE_JINAAI_512 = 'jinaai/jina-embeddings-v2-small-en'
    LATE_COLBERT_128 = 'colbert-ir/colbertv2.0'

In [18]:
class Collections(StrEnum):
    CITIES_POI = "cities_poi"

In [19]:
class CollectionVectorType(StrEnum): 
    MAIN_VECTOR = 'dense'
    LATE_VECTOR = 'late_interaction'

In [20]:
class HierarchyLevel(StrEnum):
    CITY = 'city'
    POI = 'poi'

In [107]:
from qdrant_client.models import (
    VectorParams, 
    Distance, 
    MultiVectorConfig, 
    MultiVectorComparator, 
    HnswConfigDiff
)

client.create_collection(
    collection_name=Collections.CITIES_POI,
    vectors_config={
        CollectionVectorType.MAIN_VECTOR: VectorParams(
            size=client.get_embedding_size(EmbeddingNames.DENSE_BAAI_384),
            distance=Distance.COSINE
        ),
        CollectionVectorType.LATE_VECTOR: VectorParams(
            size=client.get_embedding_size(EmbeddingNames.LATE_COLBERT_128), 
            distance=Distance.COSINE,
            multivector_config=MultiVectorConfig(comparator=MultiVectorComparator.MAX_SIM),
            hnsw_config=HnswConfigDiff(m=0)
        ),
    },
)

True

In [21]:
client.get_collection(Collections.CITIES_POI).status

<CollectionStatus.GREEN: 'green'>

### Upset embedded documents

In [22]:
from pydantic import BaseModel

class WikiCity(BaseModel): 
    city: str
    country: str
    lat: float 
    lng: float 
    population: float
    abstract: str 

    def to_searchable_text(self) -> str:
        return f"{self.city}, {self.country}. {self.abstract}"

    def to_payload(self) -> dict:
        return {
            "city": self.city,
            "country": self.country,
            "latitude": self.lat,
            "longitude": self.lng,
            "population": self.population,
            "level": HierarchyLevel.CITY,
        }

In [23]:
from typing import cast
wiki_no_embed = [WikiCity(**cast(dict, wiki_document)) for wiki_document in wiki_voyage_eu]

In [24]:
from typing import Generic, TypeVar, TypedDict 

DataT = TypeVar("DataT")

class MetaData(TypedDict):
    source: str

class DataWithMeta(BaseModel, Generic[DataT]):
    data: DataT
    metadata: MetaData

In [25]:
wiki_with_meta = [DataWithMeta[WikiCity](data=wiki, metadata={ "source": DatasetSrc.WIKI_VOYAGE_EU }) for wiki in wiki_no_embed]

In [26]:
from fastembed import TextEmbedding, LateInteractionTextEmbedding

dense_model = TextEmbedding(model_name=EmbeddingNames.DENSE_BAAI_384)
interaction_model = LateInteractionTextEmbedding(model_name=EmbeddingNames.LATE_COLBERT_128) 


In [27]:
from pathlib import Path 
import numpy as np


dense_path = Path("./embeddings/dense_embeddings_city.npy")
interaction_path = Path("./embeddings/interaction_embeddings_city.npy")

if dense_path.exists() and interaction_path.exists():
    dense = np.load(dense_path)
    interaction = np.load(interaction_path, allow_pickle=True)
else:
    dense_embdedding= list(dense_model.embed([doc.to_searchable_text() for doc in wiki_no_embed]))
    interaction_embedding = list(interaction_model.embed(doc.to_searchable_text() for doc in wiki_no_embed))
    dense = np.array(dense_embdedding)
    interaction = np.array(interaction_embedding, dtype=object)

    dense_path.parent.mkdir(parents=True, exist_ok=True)

    np.save(dense_path, dense)
    np.save(interaction_path, interaction)


In [28]:
from qdrant_client.models import PointStruct

dense_emb_list = list(dense)
interaction_emb_list = list(interaction)

points = [
    PointStruct(
        id=idx, 
        vector={
            CollectionVectorType.MAIN_VECTOR: dense_el.tolist(),
            CollectionVectorType.LATE_VECTOR: interaction_el.tolist()
        },
        payload={**doc.data.to_payload(), 'metadata': doc.metadata},

    ) for idx, (doc, dense_el, interaction_el) in enumerate(zip(wiki_with_meta, dense_emb_list, interaction_emb_list))     
]

points[:10]

[PointStruct(id=0, vector={'dense': [0.024072201922535896, 0.00202933163382113, -0.008440378122031689, -0.015731394290924072, 0.043209001421928406, -0.0022233491763472557, 0.0234548207372427, 0.04637482389807701, -0.005552217829972506, -0.005829719360917807, 0.024958105757832527, -0.0766163095831871, -0.006602343171834946, 0.02364254742860794, 0.009233275428414345, 0.003587334882467985, -0.032495226711034775, 0.016109274700284004, 0.009873833507299423, 0.011925931088626385, -0.014044007286429405, 0.00895603559911251, 0.002428129082545638, -0.06487686932086945, 0.01891295611858368, 0.03390106186270714, 0.0009602277423255146, -0.021242128685116768, -0.06506253033876419, -0.17848949134349823, 0.0353318527340889, -0.021251969039440155, 0.031347859650850296, -0.00583220599219203, -0.017568008974194527, 0.019861672073602676, -0.01798819936811924, -0.0039961389265954494, -0.017500367015600204, 0.024885065853595734, -0.00790650025010109, 0.03547412529587746, -0.0038634417578577995, 0.003905884

In [159]:
client.upsert(
    collection_name=Collections.CITIES_POI,
    points=points
)

UpdateResult(operation_id=75, status=<UpdateStatus.COMPLETED: 'completed'>)

In [120]:
import pandas as pd


voyage_df = pd.read_csv(DatasetSrc.LOCAL_VOYAGE_LISTINGS,  encoding="utf-8")

voyage_df

/var/folders/6r/698vd3f90_j2vpfcm_2r1smw0000gn/T/ipykernel_47951/3493207604.py:4: DtypeWarning: Columns (0: latitude, 1: longitude) have mixed types. Specify dtype option on import or set low_memory=False.
  voyage_df = pd.read_csv(DatasetSrc.LOCAL_VOYAGE_LISTINGS,  encoding="utf-8")


,article,type,title,alt,wikidata,wikipedia,address,directions,phone,tollFree,...,checkIn,checkOut,image,price,latitude,longitude,wifi,accessibility,lastEdit,description
0,'s-Hertogenbosch,buy,Taxi TCO,NaN,NaN,NaN,NaN,NaN,+31 412 484 41,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015-03-01,NaN
1,'s-Hertogenbosch,buy,Taxi de Hart,NaN,NaN,NaN,NaN,NaN,+31 73 5112733,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015-03-01,NaN
2,'s-Hertogenbosch,see,Saint John's Cathedral,Sint Jans Kathedraal,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,51.68808,5.30814,NaN,NaN,2016-01-25,one of the most prominent landmarks of Den Bos...
3,'s-Hertogenbosch,see,The Moriaan,NaN,NaN,NaN,NaN,on the market square,NaN,NaN,...,NaN,NaN,NaN,NaN,51.68967,5.30261,NaN,NaN,2016-01-25,"the oldest brick building in The Netherlands, ..."
4,'s-Hertogenbosch,see,Town Hall,Stadhuis,NaN,NaN,Markt 1,south side of the market square,NaN,NaN,...,NaN,NaN,NaN,NaN,51.68846,5.30315,NaN,NaN,2016-01-25,The facade was built in the 17th century and r...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
280785,Kornati National Park,see,Ilirske gradine,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2016-10-19,Remains of Illyrian settlements are located on...
280786,Kornati National Park,see,Mana,island of Mana,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,43.801115,15.263143,NaN,NaN,2016-10-19,Quite a curiosity as there in the late 50s was...
280787,Malá Fatra,see,Terchová,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,49.255556,19.033333,NaN,NaN,2016-10-18,"A tourist center of Malá Fatra, birth place of..."
280788,Malá Fatra,do,Veľký Rozsutec,NaN,NaN,NaN,NaN,Most common access is from village Štefanová (...,NaN,NaN,...,NaN,NaN,NaN,NaN,49.231944,19.100833,NaN,NaN,2016-10-18,"Measuring 1610m, the mountain provides great v..."


In [121]:
nan_stats = pd.DataFrame(
    {
        "nan_count": voyage_df.isna().sum(),
        "total": len(voyage_df)
    }
)

nan_stats["nan_ratio"] = nan_stats["nan_count"] / nan_stats["total"]

nan_stats

,nan_count,total,nan_ratio
article,0,280790,0.000000
type,70,280790,0.000249
title,511,280790,0.001820
alt,243127,280790,0.865868
wikidata,278134,280790,0.990541
wikipedia,280790,280790,1.000000
address,91880,280790,0.327220
directions,207548,280790,0.739157
phone,122724,280790,0.437067
tollFree,273762,280790,0.974971


### Preprocessing of the wikivoyage listing


We will remove some of the most useless columns

In [122]:
from pandas import DataFrame


keep_cols = ["article", "type", "title", "description", 
            "price", "latitude", "longitude", "address", "url", "hours"]


voyage_df = voyage_df.dropna(subset=["description"], inplace=False)

voyage_df= voyage_df[keep_cols]

if not isinstance(voyage_df, DataFrame):
    raise RuntimeError(f"Should be of the type dataframe instead of {voyage_df.__class__}")

else:
    voyage_df = cast(DataFrame, voyage_df) 
voyage_df

,article,type,title,description,price,latitude,longitude,address,url,hours
2,'s-Hertogenbosch,see,Saint John's Cathedral,one of the most prominent landmarks of Den Bos...,NaN,51.68808,5.30814,NaN,NaN,NaN
3,'s-Hertogenbosch,see,The Moriaan,"the oldest brick building in The Netherlands, ...",NaN,51.68967,5.30261,NaN,NaN,NaN
4,'s-Hertogenbosch,see,Town Hall,The facade was built in the 17th century and r...,NaN,51.68846,5.30315,Markt 1,NaN,NaN
5,'s-Hertogenbosch,see,The North Brabant Museum,houses a collection of art and historical arti...,NaN,51.68658,5.30469,NaN,http://www.hetnoordbrabantsmuseum.nl/english,NaN
6,'s-Hertogenbosch,see,City Museum 's-Hertogenbosch,A brand new building and a museum for modern a...,NaN,51.68575,5.30419,NaN,http://www.sm-s.nl/,NaN
...,...,...,...,...,...,...,...,...,...,...
280785,Kornati National Park,see,Ilirske gradine,Remains of Illyrian settlements are located on...,NaN,NaN,NaN,NaN,NaN,NaN
280786,Kornati National Park,see,Mana,Quite a curiosity as there in the late 50s was...,NaN,43.801115,15.263143,NaN,NaN,NaN
280787,Malá Fatra,see,Terchová,"A tourist center of Malá Fatra, birth place of...",NaN,49.255556,19.033333,NaN,http://www.terchova.sk,NaN
280788,Malá Fatra,do,Veľký Rozsutec,"Measuring 1610m, the mountain provides great v...",NaN,49.231944,19.100833,NaN,NaN,NaN


In [123]:
voyage_df[voyage_df['article'] == wiki_no_embed[0].city], wiki_no_embed[0]

(     article   type                            title  \
 92   Aalborg    see                  Aalborg Akvavit   
 93   Aalborg    see                Aalborghus Castle   
 95   Aalborg    see                 Aalborg Townhall   
 96   Aalborg    see                      Aalborg Zoo   
 100  Aalborg    see                          Elbjørn   
 101  Aalborg    see               Jens Bangs Stenhus   
 102  Aalborg    see            Jørgen Olufsens House   
 104  Aalborg    see                   Lille Vildmose   
 105  Aalborg    see                   Royal Taxhouse   
 106  Aalborg    see                     Utzon Centre   
 108  Aalborg     do                 Aalborg Karnival   
 109  Aalborg     do                          Bicycle   
 110  Aalborg     do                   Casino Aalborg   
 111  Aalborg     do                           Egholm   
 113  Aalborg     do                        Jumboland   
 115  Aalborg    buy           Algade and Bispensgade   
 116  Aalborg    buy           

In [138]:
cities = [wiki_city.city for wiki_city in wiki_no_embed]
eu_df = voyage_df[voyage_df['article'].isin(cities)]
eu_df = eu_df.dropna(subset=['description'])

In [139]:
eu_df = cast(DataFrame,eu_df)

In [141]:
len(eu_df), eu_df.isna().any()

(7158,
 article        False
 type            True
 title           True
 description    False
 price           True
 latitude        True
 longitude       True
 address         True
 url             True
 hours           True
 dtype: bool)

In [127]:
wiki_no_embed[0].city

'Aalborg'

In [144]:

eu_df["text"] = (
    eu_df["title"].fillna("")   # pyright: ignore[reportAttributeAccessIssue]
    + " — " + eu_df["type"].fillna("")  # pyright: ignore[reportAttributeAccessIssue]
    + " in " + eu_df["article"].fillna("")  # pyright: ignore[reportAttributeAccessIssue]
    + ". " + eu_df["description"]
)

eu_df['text'] = eu_df['text'].str[:256]

In [147]:
assert eu_df['text'].isna().any() == np.False_

In [148]:
articles = set(eu_df['article'])

print(len(articles))

articles

143


{'Aalborg',
 'Adana',
 'Amsterdam',
 'Ancona',
 'Ankara',
 'Antalya',
 'Arkhangelsk',
 'Astrakhan',
 'Baia Mare',
 'Baku',
 'Barcelona',
 'Bari',
 'Belgrade',
 'Bergen',
 'Berlin',
 'Bologna',
 'Bordeaux',
 'Braga',
 'Bratislava',
 'Bremen',
 'Brno',
 'Brussels',
 'Budapest',
 'Burgas',
 'Bursa',
 'Bydgoszcz',
 'Cagliari',
 'Cheboksary',
 'Chelyabinsk',
 'Cluj-Napoca',
 'Coimbra',
 'Copenhagen',
 'Cork',
 'Craiova',
 'Debrecen',
 'Denizli',
 'Dijon',
 'Donetsk',
 'Dresden',
 'Dublin',
 'Erfurt',
 'Erzurum',
 'Gaziantep',
 'Geneva',
 'Hamburg',
 'Helsinki',
 'Innsbruck',
 'Ioannina',
 'Ivano-Frankivsk',
 'Izmir',
 'Kaliningrad',
 'Kars',
 'Kaunas',
 'Kayseri',
 'Kazan',
 'Kharkiv',
 'Kiel',
 'Kirov',
 'Klagenfurt',
 'Konya',
 'Krasnodar',
 'Kutaisi',
 'Lille',
 'Ljubljana',
 'London',
 'Luxembourg',
 'Lviv',
 'Lyon',
 'Maastricht',
 'Madrid',
 'Magdeburg',
 'Malatya',
 'Milan',
 'Minsk',
 'Miskolc',
 'Moscow',
 'Munich',
 'Murmansk',
 'Nantes',
 'Naples',
 'Nevsehir',
 'Nicosia',
 'Novi

In [39]:
eu_articles = set(eu_df['article'])

print(len(eu_articles))

eu_articles

143


{'Aalborg',
 'Adana',
 'Amsterdam',
 'Ancona',
 'Ankara',
 'Antalya',
 'Arkhangelsk',
 'Astrakhan',
 'Baia Mare',
 'Baku',
 'Barcelona',
 'Bari',
 'Belgrade',
 'Bergen',
 'Berlin',
 'Bologna',
 'Bordeaux',
 'Braga',
 'Bratislava',
 'Bremen',
 'Brno',
 'Brussels',
 'Budapest',
 'Burgas',
 'Bursa',
 'Bydgoszcz',
 'Cagliari',
 'Cheboksary',
 'Chelyabinsk',
 'Cluj-Napoca',
 'Coimbra',
 'Copenhagen',
 'Cork',
 'Craiova',
 'Debrecen',
 'Denizli',
 'Dijon',
 'Donetsk',
 'Dresden',
 'Dublin',
 'Erfurt',
 'Erzurum',
 'Gaziantep',
 'Geneva',
 'Hamburg',
 'Helsinki',
 'Innsbruck',
 'Ioannina',
 'Ivano-Frankivsk',
 'Izmir',
 'Kaliningrad',
 'Kars',
 'Kaunas',
 'Kayseri',
 'Kazan',
 'Kharkiv',
 'Kiel',
 'Kirov',
 'Klagenfurt',
 'Konya',
 'Krasnodar',
 'Kutaisi',
 'Lille',
 'Ljubljana',
 'London',
 'Luxembourg',
 'Lviv',
 'Lyon',
 'Maastricht',
 'Madrid',
 'Magdeburg',
 'Malatya',
 'Milan',
 'Minsk',
 'Miskolc',
 'Moscow',
 'Munich',
 'Murmansk',
 'Nantes',
 'Naples',
 'Nevsehir',
 'Nicosia',
 'Novi

In [149]:
import numpy as np
from fastembed import TextEmbedding, LateInteractionTextEmbedding
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm


BATCH_SIZE = 16  

In [150]:
from fastembed.common.types import NumpyArray
import os


def embed_in_batches(model: TextEmbedding | LateInteractionTextEmbedding, texts: list[str], batch_size: int, desc="Embedding"):
    """Embed texts in batches with progress bar. Returns list of arrays."""
    results: list[NumpyArray] = []
    total = len(texts) 
    print(f"Batch size is {batch_size}")
    for i in tqdm(range(0, total, batch_size), desc=desc):
        batch = texts[i : i + batch_size]
        batch_embeddings = list(model.embed(batch, parallel=os.cpu_count()))
        results.extend(batch_embeddings)

    return results

In [151]:
def embed_sequential(texts: list[str], dense_model: TextEmbedding, colbert_model: LateInteractionTextEmbedding, batch_size: int):
    """
    Run dense then ColBERT sequentially. 
    Safer on RAM, use this if parallel version causes memory pressure.
    """

    dense_results = embed_in_batches(dense_model, texts, desc="Dense embedding", batch_size=batch_size)
    colbert_results = embed_in_batches(colbert_model, texts, desc="ColBERT embedding", batch_size=batch_size) 
    return dense_results, colbert_results


In [152]:
from fastembed.text import text_embedding

texts = eu_df["text"].fillna("").astype(str).tolist()

dense_embdedding_pois, interaction_embedding_pois = embed_sequential(
    texts=texts, 
    dense_model=dense_model,
    colbert_model=interaction_model,
    batch_size=BATCH_SIZE
)

Batch size is 16


Dense embedding:   0%|          | 0/448 [00:00<?, ?it/s]

Dense embedding: 100%|██████████| 448/448 [00:58<00:00,  7.62it/s]


Batch size is 16


ColBERT embedding: 100%|██████████| 448/448 [03:34<00:00,  2.09it/s]


In [153]:
len(dense_embdedding_pois), len(dense_embdedding_pois)

(7158, 7158)

In [ ]:
dense_path_poi = Path("./embeddings/dense_embeddings_poi.npy")
interaction_path_poi = Path("./embeddings/interaction_embeddings_poi.npy")

if dense_path_poi.exists() and interaction_path_poi.exists():
    dense = np.load(dense_path_poi)
    interaction = np.load(interaction_path_poi, allow_pickle=True)
else:
    texts = eu_df["text"].fillna("").astype(str).tolist()
    dense_embdedding_pois, interaction_embedding_pois = embed_sequential(
        texts=texts, 
        dense_model=dense_model,
        colbert_model=interaction_model,
        batch_size=BATCH_SIZE
    )
    dense = np.array(dense_embdedding_pois)
    interaction = np.array(interaction_embedding_pois, dtype=object)

    dense_path.parent.mkdir(parents=True, exist_ok=True)

    np.save(dense_path_poi, dense)
    np.save(interaction_path_poi, interaction)


In [156]:
from qdrant_client.models import PointStruct

dense_emb_list = list(dense)
interaction_emb_list = list(interaction)

assert len(eu_df) == len(texts)

points = [
    PointStruct(
        id=idx + len(wiki_no_embed), 
        vector={
            CollectionVectorType.MAIN_VECTOR: dense_el.tolist(),
            CollectionVectorType.LATE_VECTOR: interaction_el.tolist(),
        },
        payload={**doc, 'level': HierarchyLevel.POI, 'metadata': { 'source': DatasetSrc.LOCAL_VOYAGE_LISTINGS  } },

    ) for idx, (doc, dense_el, interaction_el) in enumerate(zip(eu_df.to_dict(orient="records"), dense_emb_list, interaction_emb_list))     
]

points[:10]

[PointStruct(id=160, vector={'dense': [0.003329081693664193, -0.03396902605891228, -0.0022375250700861216, -0.02339388057589531, -0.0019417251460254192, -0.01162264309823513, 0.06426730751991272, 0.040158115327358246, -0.01804344356060028, -0.03998682275414467, 0.011479867622256279, -0.06465984880924225, -0.04406428337097168, 0.034636225551366806, 0.033569514751434326, -0.0046084122732281685, -0.009334417060017586, 0.009203757159411907, -0.05903799831867218, 0.007337961811572313, 0.02389446832239628, -0.04417853057384491, 0.007071259897202253, -0.02864326536655426, 0.008658883161842823, 0.011083411984145641, -0.003961557522416115, -0.004248219542205334, -0.03480584919452667, -0.1333199292421341, -0.005666410084813833, -0.026964286342263222, -0.048088520765304565, 0.014285941608250141, -0.05908750370144844, 0.014237700961530209, -0.04029526934027672, 0.0018435271922498941, -0.04457001015543938, 0.02931908518075943, 0.04739309847354889, -0.002379446988925338, -0.03668399527668953, 0.0197

In [157]:
def batch(iterable, size):
    for i in range(0, len(iterable), size):
        yield iterable[i:i + size]


for batch_points in batch(points, 100): 
    client.upsert(
        collection_name=Collections.CITIES_POI,
        points=batch_points
    )

In [163]:
## Try search with reranking

from qdrant_client.conversions.common_types import QueryResponse


def simple_search(text: str, limit: int = 50) -> QueryResponse:
    # first create simple embedding 
    dense_query = list(dense_model.embed([text]))

    # then we apply search api 
    result = client.query_points(
        Collections.CITIES_POI,
        query=dense_query[0],
        using=CollectionVectorType.MAIN_VECTOR,
        with_vectors=True,
        with_payload=True,
        limit=limit
    )
    return result


In [164]:
from qdrant_client.models import Prefetch, Filter, FieldCondition, MatchValue

def search_explore(text: str, city: str, limit: int = 10, prefetch_limit=50) -> QueryResponse:
    dense_query = np.array(dense_model.embed([text]))
    colbert_query = np.array(interaction_model.embed([text]))

    result = client.query_points(
        Collections.CITIES_POI,
        prefetch=Prefetch(
            query=dense_query[0],
            using=CollectionVectorType.MAIN_VECTOR,
            limit=prefetch_limit,
            filter=Filter(
                must=[
                    FieldCondition(
                        key="city",
                        match=MatchValue(value=city),
                    ),
                    FieldCondition(
                        key="level",
                        match=MatchValue(value=HierarchyLevel.POI),
                    ),
                ]
            ),
        ),
        query=colbert_query,
        using=CollectionVectorType.LATE_VECTOR
    )
    return result

In [171]:
from typing import OrderedDict
from qdrant_client.models import ScoredPoint

def search_discover(text: str, limit: int = 10, prefetch_limit=50, poi_threshold=0.5) -> dict[str, list[ScoredPoint]]: 
    dense_query = list(dense_model.embed([text]))[0]                                                                                                            
    colbert_query = list(interaction_model.embed([text]))[0] 

    result = client.query_points(
        Collections.CITIES_POI,
        prefetch=Prefetch(
            query=dense_query.tolist(),
            using=CollectionVectorType.MAIN_VECTOR,
            limit=prefetch_limit,
        ),
        query=colbert_query.tolist(),
        using=CollectionVectorType.LATE_VECTOR,
        limit=limit,
    )
    cities: set[str] = (set([
        city.payload['city'] for city in result.points 
            if city.payload is not None and city.payload['level'] == HierarchyLevel.CITY
    ]))

    pois = [poi for poi in result.points 
            if poi.payload is not None and poi.payload['level'] == HierarchyLevel.POI
    ]
    poi_cities = (set([poi.payload['article'] for poi in pois if poi.payload is not None]))

    grouped: OrderedDict[str, list[ScoredPoint]] = OrderedDict()

    for point in result.points:
        if point.payload is None:
            raise ValueError("Payload should contain actual values")
        
        if point.payload["level"] != HierarchyLevel.POI:
            continue
        
        article = point.payload["article"]
        
        # Only include POIs whose city is recognized
        if len(cities) / len(poi_cities) >= poi_threshold and article not in cities:
            continue

        if article not in grouped:
            grouped[article] = []  # first POI for this city — determines city rank
        grouped[article].append(point)
    
    return grouped

In [177]:
def search_discover_bottom_up(query, limit=20):
    dense_query = list(dense_model.embed([query]))[0].tolist()
    colbert_query = list(interaction_model.embed([query]))[0].tolist()

    result = client.query_points_groups(
        collection_name=Collections.CITIES_POI,
        prefetch=Prefetch(
            query=dense_query, 
            using=CollectionVectorType.MAIN_VECTOR, 
            limit=100,
            filter=Filter(must=[
                FieldCondition(key="level", match=MatchValue(value="poi"))
            ]),
        ),
        query=colbert_query, 
        using=CollectionVectorType.LATE_VECTOR,
        group_by="article",
        group_size=5,
        limit=limit,
    )
    return result

In [185]:
def search_discover_top_down(query, top_cities=3, pois_per_city=5):
    dense_query = list(dense_model.embed([query]))[0].tolist()
    colbert_query = list(interaction_model.embed([query]))[0].tolist()

    # Stage 1: find best matching cities
    cities_result = client.query_points(
        collection_name=Collections.CITIES_POI,
        prefetch=Prefetch(
            query=dense_query, 
            using=CollectionVectorType.MAIN_VECTOR, 
            limit=20,
            filter=Filter(must=[
                FieldCondition(key="level", match=MatchValue(value="city"))
            ]),
        ),
        query=colbert_query, 
        using=CollectionVectorType.LATE_VECTOR,
        limit=top_cities,
    )

    # Stage 2: for each city, find best POIs
    results = {}
    for city_point in cities_result.points:
        if city_point.payload is None:
            raise ValueError("Payload is required!")
        city_name = city_point.payload["city"]
        pois = client.query_points(
            collection_name=Collections.CITIES_POI,
            prefetch=Prefetch(
                query=dense_query, 
                using=CollectionVectorType.MAIN_VECTOR, 
                limit=30,
                filter=Filter(must=[
                    FieldCondition(key="article", match=MatchValue(value=city_name)),
                    FieldCondition(key="level", match=MatchValue(value="poi")),
                ]),
            ),
            query=colbert_query, 
            using=CollectionVectorType.LATE_VECTOR,
            limit=pois_per_city,
        )
        results[city_name] = pois.points

    return cities_result.points, results

In [161]:
query = "sea and ocean"

In [166]:
results = simple_search(query)

In [167]:
results.points[0]

ScoredPoint(id=2635, version=100, score=0.87803084, payload={'article': 'Kaliningrad', 'type': 'do', 'title': 'Fish at the Baltic sea', 'description': 'All year round', 'price': None, 'latitude': None, 'longitude': None, 'address': None, 'url': None, 'hours': None, 'text': 'Fish at the Baltic sea — do in Kaliningrad. All year round', 'level': 'poi', 'metadata': {'source': '../datasets/wikivoyage-listings-en.csv'}}, vector={'late_interaction': [[-0.004218309, -0.097316764, 0.082310125, -0.12604204, 0.0017511696, 0.008383183, -0.013370928, 0.13187975, -0.090669215, -0.018062467, 0.03852525, -0.042620413, -0.0904359, -0.20523854, -0.05119867, -0.049445655, -0.08538485, 0.03416165, 0.12534074, 0.14341058, 0.064061224, 0.002086616, 0.10661994, -0.081941135, -0.06745167, 0.068063825, -0.06960214, -0.0038540754, 0.11401802, -0.017878717, 0.0735895, -0.09701547, 0.16796754, 0.103779234, -0.07808733, -0.029087583, -0.03740878, 0.20692815, 0.03530605, -0.021458695, 0.058219887, -0.07557938, 0.10

In [168]:
for point in results.points:
    if point.payload is None: 
        raise ValueError("Payload is not found")
    print(f"{point.score:.4f} | {point.payload.get('city', point.payload.get('article'))} | {point.payload.get('level')}")

0.8780 | Kaliningrad | poi
0.8470 | Samsun | poi
0.8463 | Baku | poi
0.8447 | Zagreb | poi
0.8349 | Burgas | poi
0.8348 | Santander | poi
0.8337 | Murcia | city
0.8333 | Valladolid | city
0.8330 | Saint Petersburg | poi
0.8310 | Santander | poi
0.8308 | Astrakhan | poi
0.8305 | Kaliningrad | poi
0.8296 | Cluj-Napoca | poi
0.8292 | Bergen | poi
0.8290 | Thessaloniki | poi
0.8285 | Sofia | poi
0.8276 | Coimbra | poi
0.8268 | Geneva | poi
0.8266 | Belgrade | poi
0.8263 | Baku | poi
0.8262 | Bergen | poi
0.8255 | Turku | poi
0.8253 | Cluj-Napoca | poi
0.8249 | Voronezh | city
0.8248 | Baku | poi
0.8247 | Saint Petersburg | poi
0.8245 | Baku | poi
0.8235 | Penza | city
0.8233 | Budapest | poi
0.8233 | Santander | poi
0.8232 | Brest | city
0.8230 | Tbilisi | poi
0.8227 | Samsun | poi
0.8226 | Tbilisi | poi
0.8224 | Baku | poi
0.8220 | Baku | poi
0.8218 | Kiel | poi
0.8215 | Lviv | poi
0.8210 | Geneva | poi
0.8209 | Naples | poi
0.8209 | Santander | poi
0.8209 | Belgrade | poi
0.8207 | Santan

In [172]:
results_late_interaction = search_discover(query, 20)

In [ ]:
results_late_interaction

# Result: reranking solved the ranking problem -> most of the cities from the land are dropped and many coastal cities rose (Kaliningrad, Bergen)
# Problems are still here: there is no semantic understanding. dataset is not clear enough we need to apply some filters to solve data quality issue

OrderedDict([('Kaliningrad',
              [ScoredPoint(id=2632, version=100, score=3.1762664, payload={'article': 'Kaliningrad', 'type': 'see', 'title': 'Museum of the World Ocean', 'description': 'Includes two museum ships and one submarine.', 'price': None, 'latitude': 54.7067, 'longitude': 20.5001, 'address': 'Nab. Petra Velikovo 1', 'url': 'http://world-ocean.ru/', 'hours': '11AM to 6PM, Wed-Sun', 'text': 'Museum of the World Ocean — see in Kaliningrad. Includes two museum ships and one submarine.', 'level': 'poi', 'metadata': {'source': '../datasets/wikivoyage-listings-en.csv'}}, vector=None, shard_key=None, order_value=None),
               ScoredPoint(id=2635, version=100, score=2.3306565, payload={'article': 'Kaliningrad', 'type': 'do', 'title': 'Fish at the Baltic sea', 'description': 'All year round', 'price': None, 'latitude': None, 'longitude': None, 'address': None, 'url': None, 'hours': None, 'text': 'Fish at the Baltic sea — do in Kaliningrad. All year round', 'level': 

In [178]:
bottom_up_discover_results = search_discover_bottom_up(query)

In [ ]:
# bottom-up approach is way better at capturing actually relevant results
for group in bottom_up_discover_results.groups:
    print(f"\n=== Group: {group.id} ===")

    for point in group.hits:   # sometimes called .points depending on version
        if point.payload is None:
            raise ValueError("Payload is not found")

        print(
            f"{point.score:.4f} | "
            f"{point.payload.get('city', point.payload.get('article'))} | "
            f"{point.payload.get('level')} | " 
            f"{point.payload.get('title')}"
        )


=== Group: Kaliningrad ===
3.1763 | Kaliningrad | poi | Museum of the World Ocean
2.3307 | Kaliningrad | poi | Fish at the Baltic sea
1.1674 | Kaliningrad | poi | Fishing Village
1.1672 | Kaliningrad | poi | Skipper Hotel
1.0223 | Kaliningrad | poi | Solyanka Cafe

=== Group: Maastricht ===
2.9567 | Maastricht | poi | Onze Lieve Vrouwebasiliek
0.9499 | Maastricht | poi | Pathé Cinema
0.9336 | Maastricht | poi | Wok to go
0.8505 | Maastricht | poi | Eetcafé De Preuverij
0.7801 | Maastricht | poi | MECC Maastricht

=== Group: Kiel ===
2.9327 | Kiel | poi | Aquarium GEOMAR
2.6602 | Kiel | poi | Marine-Ehrenmal
2.4267 | Kiel | poi | Zoologisches Museu
1.6956 | Kiel | poi | Type VII-C U-boat
1.2901 | Kiel | poi | Hotel Kieler Yacht Club

=== Group: Miskolc ===
2.7820 | Miskolc | poi | Pannon-sea Museum
1.3839 | Miskolc | poi | The Cave Bath
1.2195 | Miskolc | poi | Cave bath
0.9957 | Miskolc | poi | Tapolca Inn
0.8915 | Miskolc | poi | Missionart Gallery

=== Group: Antalya ===
2.6144 | An

In [186]:
## two stage approach
top_down_discover_results = search_discover_top_down(query)

In [188]:
top_down_discover_results

([ScoredPoint(id=63, version=75, score=2.2047377, payload={'city': 'Kiel', 'country': 'Germany', 'latitude': 54.3233, 'longitude': 10.1394, 'population': 246601.0, 'level': 'city', 'metadata': {'source': 'ashmib/wikivoyage-eu-city-embeddings'}}, vector=None, shard_key=None, order_value=None),
  ScoredPoint(id=3, version=75, score=2.1787705, payload={'city': 'Ancona', 'country': 'Italy', 'latitude': 43.6169, 'longitude': 13.5167, 'population': 100924.0, 'level': 'city', 'metadata': {'source': 'ashmib/wikivoyage-eu-city-embeddings'}}, vector=None, shard_key=None, order_value=None),
  ScoredPoint(id=116, version=75, score=1.9807749, payload={'city': 'Samsun', 'country': 'Turkey', 'latitude': 41.2903, 'longitude': 36.3336, 'population': 1335716.0, 'level': 'city', 'metadata': {'source': 'ashmib/wikivoyage-eu-city-embeddings'}}, vector=None, shard_key=None, order_value=None)],
 {'Kiel': [ScoredPoint(id=3023, version=104, score=2.9327202, payload={'article': 'Kiel', 'type': 'see', 'title': '

In [204]:
## implementing encoder based architecture 
from fastembed.rerank.cross_encoder import TextCrossEncoder

cross_encoder = TextCrossEncoder('Xenova/ms-marco-MiniLM-L-6-v2')



def search_with_encoding(query: str, top_cities=5, pois_per_city=5):
    dense_query = list(dense_model.embed([query]))[0].tolist()
    cities_result = client.query_points(
        collection_name=Collections.CITIES_POI,
        query=dense_query,
        using=CollectionVectorType.MAIN_VECTOR,
        limit=top_cities,
        query_filter=Filter(must=[
            FieldCondition(key="level", match=MatchValue(value=HierarchyLevel.CITY))
        ]),
    )
    results = {}

    city_names = set(p.payload["city"] for p in cities_result.points if p.payload)

    # What articles actually exist in the POI data?
    for city in city_names:
        count = client.count(
            collection_name=Collections.CITIES_POI,
            count_filter=Filter(must=[
                FieldCondition(key="article", match=MatchValue(value=city)),
                FieldCondition(key="level", match=MatchValue(value="poi")),
            ]),
        )
        print(f"{city}: {count.count} POIs")

    # for each city make cross-encoding
    for city_point in cities_result.points:
        if city_point.payload is None:
            raise ValueError("City payload cannot be None")
        city_name = city_point.payload['city']

        candidates = client.query_points(
            collection_name=Collections.CITIES_POI,
            query=dense_query,
            using=CollectionVectorType.MAIN_VECTOR,
            limit=40,
            query_filter=Filter(must=[
                FieldCondition(key="article", match=MatchValue(value=city_name)),
                FieldCondition(key="level", match=MatchValue(value="poi")),
            ]),
        )

        # actual cross-encoding scoring 
        candidates_texts: list[str] = [p.payload['text'] for p in candidates.points if p.payload is not None]
        scores = list(cross_encoder.rerank(query, candidates_texts))

        reranked = sorted(
            zip(candidates.points, scores),
            key=lambda x: x[1],
            reverse=True,
        )[:pois_per_city]
        results[city_name] = reranked
    return results

In [210]:
from fastembed.rerank.cross_encoder import TextCrossEncoder

cross_encoder = TextCrossEncoder('Xenova/ms-marco-MiniLM-L-6-v2')

def search_with_cross_encoder(query: str, prefetch_limit=100, final_limit=10):
    dense_query = list(dense_model.embed([query]))[0].tolist()

    candidates = client.query_points(
        collection_name=Collections.CITIES_POI,
        query=dense_query,
        using=CollectionVectorType.MAIN_VECTOR,
        limit=prefetch_limit,
        query_filter=Filter(must=[
            FieldCondition(key="level", match=MatchValue(value="poi")),
        ]),
    )

    candidate_texts = [p.payload['text'] for p in candidates.points if p.payload]
    scores = list(cross_encoder.rerank(query, candidate_texts))

    reranked = sorted(
        zip(candidates.points, scores),
        key=lambda x: x[1],
        reverse=True,
    )[:final_limit]
    return reranked

In [211]:
result_encoder = search_with_cross_encoder(query)

In [212]:
result_encoder

[(ScoredPoint(id=2632, version=100, score=0.83048666, payload={'article': 'Kaliningrad', 'type': 'see', 'title': 'Museum of the World Ocean', 'description': 'Includes two museum ships and one submarine.', 'price': None, 'latitude': 54.7067, 'longitude': 20.5001, 'address': 'Nab. Petra Velikovo 1', 'url': 'http://world-ocean.ru/', 'hours': '11AM to 6PM, Wed-Sun', 'text': 'Museum of the World Ocean — see in Kaliningrad. Includes two museum ships and one submarine.', 'level': 'poi', 'metadata': {'source': '../datasets/wikivoyage-listings-en.csv'}}, vector=None, shard_key=None, order_value=None),
  -3.7607762813568115),
 (ScoredPoint(id=3936, version=113, score=0.81891143, payload={'article': 'Minsk', 'type': 'do', 'title': 'Minsk Sea', 'description': "This  is an artificial reservoir.  There's a free public beach, and pedal-boat and catamaran rental.", 'price': None, 'latitude': None, 'longitude': None, 'address': None, 'url': None, 'hours': None, 'text': "Minsk Sea — do in Minsk. This  i

In [187]:
type(results.points)

list

## Remove collection

In [93]:
client.delete_collection(Collections.CITIES_POI)

False